# Init Icechunk Repo and Add Data
This notebook creates an Icechunk repository in a GCS bucket, copies the contents of another zarr from another bucket location, and commits the changes to the Icechunk store. 

Icechunk allows Zarr stores to be treated kind of like GitHub repositories. There is a main branch, from which other branches can be made. Changes can be made to the main branch or other branches and committed. Once changes are committed, it creates an immutable `snapshot`. A snapshot is a file that lives in the repo (zarr) that provides references to regions in the chunk manifest files that have changed. 

For more info, see: https://icechunk.io/en/latest/spec/#overview

### Icechunk repo files             
`chunks/` -- The actual data          
`manifests/` -- Descriptors of the data          
`refs/` -- Tracks the state of branches and tags          
`snapshots/` -- describes committed changes to the repo          
`transactions/` -- Contains operations that were done to the repo. Used for rebase and diffs      

### How Icechunk handles read/write
Taken from: https://icechunk.io/en/latest/spec/#specification  
When reading from object store, the client opens the latest branch or tag file to obtain a pointer to the relevant snapshot file. The client then reads the snapshot file to determine the structure and hierarchy of the repository. When fetching data from an array, the client first examines the chunk manifest file[s] for that array and finally fetches the chunks referenced therein.

When writing a new repository snapshot, the client first writes a new set of chunks and chunk manifests, and then generates a new snapshot file. Finally, to commit the transaction, it updates the branch reference file using an atomic conditional update operation. This operation may fail if a different client has already committed the next snapshot. In this case, the client may attempt to resolve the conflicts and retry the commit.

In [1]:
import icechunk
import zarr
import numpy as np

## Connect to data source
Opens cloud storage backend for GCS as a store instance. 

In [2]:
storage = icechunk.gcs_storage(bucket="aifi-static-assets-prod-public", prefix="dev/icechunk-test", from_env=True)

In [3]:
type(storage)

Storage

### Create a new empty repo or open existing
If `storage` is already an Icechunk repository, it will be opened. If not, a new repository will be created at the prefix specified in `storage`.        
NOTE: Icechunk repositories can only be created in a clean prefix. There is no "git init" equivalent for Icechunk.  

In [4]:
# create new or open existing
repo = icechunk.Repository.open_or_create(storage)

## Create or checkout a branch and make writable

In [5]:
repo.list_branches()

{'add_data', 'add_sc_data', 'main'}

Select a branch to use from above, or make a new one. You "checkout" a new/existing branch by using the `repo.writable_session()` method, which creates or opens a branch (zarr store) that can be written to, or `repo.readonly_session()`, which checks out a new/existing branch that is read only. These operations return a `session` object, which is the branch itself which has the underlying zarr store as an atrribute. 

In [6]:
branch_name = 'add_sc_data'

In [21]:
if branch_name in repo.list_branches():
    session = repo.writable_session(branch_name) 
    print(f'Checkout to branch: {branch_name}')
else:
    # create branch from main
    hist = repo.ancestry(branch="main")
    snapshot_id_to_branch_from = list(hist)[0].id
    repo.create_branch("add_sc_data", snapshot_id=snapshot_id_to_branch_from)
    session = repo.writable_session(branch_name) 
    print(f'New branch "{branch_name}" initiated from "main" snapshot {snapshot_id_to_branch_from}.')

Checkout to branch: add_sc_data


## Open zarr     
`session.store` can be treated like a normal zarr store, though it will have type `icechunk.store.IcechunkStore`. But it seems to pass all type checks for `zarr.core.Group`. 
It can be passed directly to `zarr.open()` and you can also use the normal zarr API to access and write data. 

In [8]:
# now get the store from this writable session so that data can be added
z = zarr.open(session.store)  # icechunk stores can be passed directly to zarr.open
z

<Group <icechunk.store.IcechunkStore object at 0x7956c2739910>>

## Copy contents of a zarr to icechunk store
Since there is no equivalent operation to `git init` in Icechunk, the contents of an existing zarr must be copied over to the new Icechunk repository. The icechunk store gets treated like a top level zarr group, and can create arrays or branching groups within it.

`src_group` is the path to a zarr that is to be copied to the Icechunk repository. 

In [9]:
src_group = zarr.open('gs://aifi-static-assets-prod-public/dev/scarf-test/data.zarr', mode="r")
src_group

<Group <FsspecStore(GCSFileSystem, aifi-static-assets-prod-public/dev/scarf-test/data.zarr)>>

## Copy the contents of an existing Zarr to the Icechunk repo

This function will recursively copy all contents (including attrs) from an existing zarr to the Icechunk store. There are some compressor issues that need to be figured out

In [10]:
# for zarr 3.X.X
from zarr.codecs import BloscCodec
compressor = BloscCodec(cname="zstd", clevel=3,shuffle='bitshuffle', blocksize=0 )

In [11]:
def recursive_copy_zarr(src_group: zarr.Group, dst_group: zarr.Group, compressor: BloscCodec = None) -> None:
    """
    Recursively copy a Zarr group and all subgroups/arrays into dst_group.
    """
    # copy store-level attrs
    dst_group.attrs.update(src_group.attrs)

    def _copy_group(src: zarr.Group, dst: zarr.Group):
        # copy group-level attrs
        dst.attrs.update(src.attrs)

        # iterate over all keys (group or array)
        for key in src.keys():
            obj = src[key]
            if isinstance(obj, zarr.Group):
                # copy sub group
                sub_dst = dst.require_group(key)
                _copy_group(obj, sub_dst)
            elif isinstance(obj, zarr.Array):
                # copy array and data
                dst_array = dst.create_array(
                    key,
                    # shape=obj.shape,
                    # dtype=obj.dtype,
                    data=obj[:],
                    chunks=obj.chunks,
                    compressors=(compressor,), # compressor is depricated, use compressor as tuple
                    fill_value=obj.fill_value,
                    overwrite=True,
                )
                dst_array.attrs.update(obj.attrs.asdict())
                #dst_array[:] = obj[:]  # eager full copy
            else:
                raise TypeError(f"Unexpected Zarr item: {obj}")

    _copy_group(src_group, dst_group)
    return 

### Use the function above to recursively copy all of the contents of `src_group` to `z` from the Open Zarr section (the new Icechunk repo store). 

In [12]:
# copy contents
recursive_copy_zarr(src_group, z, compressor)

/home/workspace/environment/minimalv2/lib/python3.11/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=18, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/workspace/environment/minimalv2/lib/python3.11/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=15, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future ve

Double check that the contents of the expression matrix are the same. 

In [13]:
z['RNA/counts'].shape == src_group['RNA/counts'].shape

True

## Commit changes to icechunk store
There is no "pull / push" equivalent in Icechunk. Any committed changes are immediately live in the specified data source. 
Commit the changes and add a commit message. 

In [14]:
snapshot_id = session.commit("commit scRNA data for real this time")
print(snapshot_id)

M69BJ0XM9NCW0WHY8ZC0


## Explore session

In [15]:
repo

See the commits to the main branch

In [16]:
hist = repo.ancestry(branch="main")
for ancestor in hist:
    print(ancestor.id, ancestor.message, ancestor.written_at)

1CECHNKREP0F1RSTCMT0 Repository initialized 2025-09-24 17:19:33.756418+00:00


See the commits to the new / selected branch

In [17]:
hist = repo.ancestry(branch=branch_name)
for ancestor in hist:
    print(ancestor.id, ancestor.message, ancestor.written_at)

M69BJ0XM9NCW0WHY8ZC0 commit scRNA data for real this time 2025-09-30 23:16:15.974237+00:00
8D4QQXXGG6A54XHV62J0 commit scRNA data for real this time 2025-09-29 19:38:53.033849+00:00
M26VAW84W9DPCJJKCX8G commit scRNA data for real this time 2025-09-26 17:24:49.933706+00:00
KV58M0NF4678FAW57NXG commit scRNA data for real this time 2025-09-25 21:59:55.708972+00:00
Y9RF5WF4DA7ESWNFZ3GG commit scRNA data for real this time 2025-09-25 21:50:46.618312+00:00
EJFM8D3NE27P2K3H3FBG commit scRNA data for real this time 2025-09-25 20:41:59.108478+00:00
1CECHNKREP0F1RSTCMT0 Repository initialized 2025-09-24 17:19:33.756418+00:00


Checkout to an earlier session (like checking out to an old git commit). 

In [18]:
earlier_session = repo.readonly_session(snapshot_id=snapshot_id)
store = earlier_session.store
group = zarr.open_group(store, mode="r")
group

<Group <icechunk.store.IcechunkStore object at 0x7956c05f0b50>>

## Merging changes to the main branch
There isn't really a "merge" function like in git (see https://icechunk.io/en/v1.0.0/icechunk-for-git-users/#merging-and-rebasing).
If you're happy with the changes in the new branch, you could reset the branch, which copes all the work from the new branch back
to main. NOTE: THIS IS A DESTRUCTIVE OPERATION. See: https://icechunk.io/en/v1.0.0/icechunk-for-git-users/#resetting-a-branch 

In [23]:
new_snapshot = list(repo.ancestry(branch=branch_name))[0].id
new_snapshot

'M69BJ0XM9NCW0WHY8ZC0'

In [24]:
repo.reset_branch("main", snapshot_id=new_snapshot)

Check that the merge worked

In [26]:
hist = repo.ancestry(branch='main')
for ancestor in hist:
    print(ancestor.id, ancestor.message, ancestor.written_at)

M69BJ0XM9NCW0WHY8ZC0 commit scRNA data for real this time 2025-09-30 23:16:15.974237+00:00
8D4QQXXGG6A54XHV62J0 commit scRNA data for real this time 2025-09-29 19:38:53.033849+00:00
M26VAW84W9DPCJJKCX8G commit scRNA data for real this time 2025-09-26 17:24:49.933706+00:00
KV58M0NF4678FAW57NXG commit scRNA data for real this time 2025-09-25 21:59:55.708972+00:00
Y9RF5WF4DA7ESWNFZ3GG commit scRNA data for real this time 2025-09-25 21:50:46.618312+00:00
EJFM8D3NE27P2K3H3FBG commit scRNA data for real this time 2025-09-25 20:41:59.108478+00:00
1CECHNKREP0F1RSTCMT0 Repository initialized 2025-09-24 17:19:33.756418+00:00


All of the changes to new_branch were now applied to the main branch. Alternatively you could skip this and just continue working from the new branch, but all users must understand that that new branch is the new "official" branch. 